# Asset Class Trend Following 策略回測 (equityV(調))

本報告針對調整後的資金與 MDD 邏輯進行回測。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from backtest_equityV_adj import clean_data, BacktesterAdjusted, calculate_metrics_adj

# 1. 資料讀取
prices, volumes, code_to_name = clean_data('樣本集-1.xlsx')

# 2. 設定參數與執行回測
sma, roc, sl, reb = 303, 14, 0.0999, 9
bt = BacktesterAdjusted(prices, volumes, code_to_name, 
                        authorized_capital=60000000, 
                        trading_capital=12000000)
eq, trades, hold, trades2, daily = bt.run(sma, roc, sl, reb)

# 3. 篩選指定期間績效
mask = (eq['日期'] >= '2019-01-01') & (eq['日期'] <= '2025-12-31')
res_p = eq[mask]

# 4. 繪製權益曲線
plt.figure(figsize=(12, 6))
plt.plot(res_p['日期'], res_p['權益'])
plt.title(f'Equity Curve (equityV(調))')
plt.grid(True)
plt.show()

# 5. 計算績效指標 (回測模式)
cagr, mdd, fmdd, calmar, total_ret = calculate_metrics_adj(res_p, 60000000)
print(f"年化報酬率 (CAGR): {cagr:.2%}")
print(f"最大回撤 (MaxDD - 投資報告標準): {mdd:.2%}")
print(f"固定基準 MDD (實際資金風險控管): {fmdd:.2%}")
print(f"Calmar Ratio: {calmar:.2f}")
print(f"總報酬率: {total_ret:.2%}")

# 6. 年度績效 (實際交易模式)
res_p['Year'] = res_p['日期'].dt.year
for year, group in res_p.groupby('Year'):
    y_ret = (group['權益'].iloc[-1] / group['權益'].iloc[0]) - 1
    y_peak = group['權益'].cummax()
    y_mdd = ((group['權益'] - y_peak) / y_peak).min()
    print(f"{year} 年度報酬: {y_ret:.2%}, 年度 MaxDD: {y_mdd:.2%}")
